# Data Refresh — operations console (generic)

Drives the broker-agnostic steps. **No broker-specific or ingestion logic lives here**
— fetching is per broker (see `broker-robinhood.ipynb`, `broker-fidelity.ipynb`); the
rest is the library.

**The loop:** ① fetch each broker (the `broker-*.ipynb` notebooks) · ② rebuild ·
③ check what reconciles · ④ curate fixups for the rest.

## Freshness — what data is the pipeline using, and how old?

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
from invest import config, broker

def _stamp(p):
    p = Path(p)
    if p.is_dir():                                  # e.g. the Robinhood history store
        env = p / "history.json"
        return pd.Timestamp(json.loads(env.read_text()).get("fetched_at")) if env.exists() else None
    m = re.match(r"(\d{8})", p.name)                # prefer a YYYYMMDD_ name
    return pd.Timestamp(m.group(1)) if m else pd.Timestamp(p.stat().st_mtime, unit="s")

def _age(ts):
    ts = pd.Timestamp(ts)
    if pd.isna(ts):
        return None
    if ts.tz is not None:                           # some stamps are tz-aware
        ts = ts.tz_convert(None)
    return (pd.Timestamp.now().normalize() - ts.normalize()).days

# Iterate the broker registry — a new broker shows up here automatically.
rows = []
for b in broker.BROKERS:
    d = config.RAW_DIR / b.subdir
    for kind, locator in (("positions", b.positions_locator or broker._latest_csv),
                          ("transactions", b.transactions_locator)):
        p = locator(d) if (locator and d.exists()) else None
        if p is not None:
            rows.append((f"{b.name} {kind}", p.name, _stamp(p)))

df = pd.DataFrame(rows, columns=["source", "file", "as_of"])
df["age_days"] = df["as_of"].apply(_age)
df.style.apply(lambda s: ["color: #c00" if v and v > 45 else "" for v in df["age_days"]], subset=["age_days"])

## ① Fetch each broker

Run the per-broker notebooks first: **`broker-robinhood.ipynb`** (token + fetch) and **`broker-fidelity.ipynb`** (manual export). Then continue here.

## ② Rebuild — regenerate the ledger, derive, reconcile

In [ ]:
import sys
!{sys.executable} -m invest.pipeline

## ③ What reconciles, and what needs attention

`reconcile` compares ledger-derived holdings to the broker snapshot per `(broker, symbol)`. Rows where `match` is False need a fixup.

In [ ]:
from invest import ledger
entries, errors, _ = ledger.load()
print(f"ledger: {len(entries)} entries, {len(errors)} load error(s)")
for e in errors[:10]:
    print("  ⚠", e.message[:100])

derived = ledger.positions_dataframe(entries)
snapshot = broker.load_all_positions(verbose=False)
rec = ledger.reconcile(derived, snapshot)
print(f"\nshares reconcile: {int(rec['match'].sum())}/{len(rec)}")
rec.round(3)

### Data quality — classification & pricing

Any holding the symbol map doesn't cover (`unclassified`), and how each holding was priced (`price_used`).

In [ ]:
positions = pd.read_parquet(config.POSITIONS_PARQUET)
unknown = positions[positions["asset_class"] == "unclassified"]
print(f"Unclassified holdings: {len(unknown)}" + ("  — add them to config/symbol_map.yaml" if len(unknown) else " ✅"))
if len(unknown):
    display(unknown[["broker", "symbol", "description", "market_value"]])
print("\nHow each holding was priced (price_used):")
display(positions.groupby("price_used")["market_value"].agg(holdings="count", value="sum").round(0))
unpriced = positions[positions["price_used"] == "unavailable"]
if len(unpriced):
    print("⚠ Unpriced (market_value NaN) — needs a yf_symbol or a snapshot price:")
    display(unpriced[["broker", "account_name", "symbol", "quantity"]])

## Opportunities dashboard

A one-glance digest across the four finders — harvestable losses, cash, asset location,
concentration risk and fund fees. Each line has its own deep-dive notebook
(`opportunities-*.ipynb`); the lot-sale planner answers "which lots do I sell to raise
cash at minimum tax?"

In [ ]:
from invest import config, ledger, mapping, opportunities as O
import pandas as pd

_pos = pd.read_parquet(config.POSITIONS_PARQUET)
_hist = pd.read_parquet(config.PRICES_PARQUET)
_txn = pd.read_parquet(config.TRANSACTIONS_PARQUET)
_ent, _, _ = ledger.load()
_meta = mapping.load_account_meta()
s = O.summary(_pos, _hist, _txn, _ent, _meta)

print("OPPORTUNITIES".center(64, "="))
print(f"  Tax-loss harvest : ${-s['harvest_loss']:,.0f} of losses in {s['harvest_lots']} taxable lots"
      + (f"  (offsets ${s['realized_ytd']:,.0f} YTD gains)" if s['realized_ytd'] else ""))
if s['wash_blocked']:
    print(f"                     wash-sale risk on: {', '.join(s['wash_blocked'])}")
print(f"  Cash             : ${s['cash_total']:,.0f} ({s['cash_pct']:.0%} of portfolio)"
      + (f"  -- ${s['idle_cash']:,.0f} idle (<1%)" if s['idle_cash'] else "  -- earning a sweep yield"))
print(f"  Asset location   : " + (f"${s['misplaced_value']:,.0f} tax-inefficient sitting in taxable"
      if s['misplaced_value'] else "tax-inefficient assets are sheltered"))
print(f"  Concentration    : {s['top_risk_name']} = {s['top_risk_share']:.0%} of risk at "
      f"{s['top_risk_weight']:.0%} weight  (largest: {s['max_name']} {s['max_name_weight']:.0%})")
print(f"  Fund fees        : ${s['expense_annual']:,.0f}/yr"
      + (f"  -- lagging benchmark: {', '.join(s['laggards'])}" if s['laggards'] else ""))
print("=" * 64)
print("Deep dives -> notebooks/opportunities-*.ipynb")

## ④ Curate `ledger/fixups.beancount`

Most gaps self-resolve via auto opening lots. The rest are judgment calls — a merger,
an un-recorded transfer, a split. Each unreconciled holding below comes with a
**starter directive** to paste into `ledger/fixups.beancount` and refine, then
re-derive (rerun ② with `--no-emit`).

In [ ]:
bad = rec[~rec["match"]]
if bad.empty:
    print("All holdings reconcile ✅ — nothing to curate.")
else:
    for (bk, sym), r in bad.iterrows():
        diff = r["derived"] - r["snapshot"]
        acct = derived[(derived["broker"] == bk) & (derived["symbol"] == sym)]["account_name"]
        acct = acct.iloc[0] if len(acct) else "?Account"
        bc = f"Assets:{bk.title()}:{acct}"
        print(f"# {bk} {sym}: derived {r['derived']:.3f} vs snapshot {r['snapshot']:.3f} (diff {diff:+.3f})")
        if diff > 0:
            print(f'2026-01-01 * "{sym} — book out residual (refine date/economics)"')
            print(f"  {bc}  {-diff:.3f} {sym} {{}}")
            print("  Equity:Adjustments\n")
        else:
            print(f'2000-01-01 * "{sym} — opening lot (refine cost)"')
            print(f"  {bc}  {-diff:.3f} {sym} {{0.00 USD}}")
            print("  Equity:Opening-Balances\n")